# ArcVox — Quick Listen  🔊  (no setup, no uploads, no keys)

**For non-technical users. Just do this:**

1. At the top menu, click **Runtime → Change runtime type → GPU → Save**.
2. Click **Runtime → Run all**.
3. Wait ~5–8 minutes (it downloads the voice model the first time).
4. Scroll down and press **▶ play** on each audio player to hear the voices.

That's it. Nothing to upload, no account, no API key. This plays an English
voice and a **Hindi** voice from the same engine ArcVox uses, then shows the
transcription engine reading them back.

> To also test **voice cloning**, **talking-head avatars**, and the full
> **21 Indian languages** (Tamil, Telugu, Bengali, …), use the bigger notebook
> `arcvox_gpu_verify.ipynb` — those need a photo/voice upload and a free
> Hugging Face token.


In [ ]:
!nvidia-smi

### Step 1 — install the voice engine (Chatterbox, ~1–2 min)

In [ ]:
!pip install -q chatterbox-tts

### Step 2 — generate an English voice and a Hindi voice
Press ▶ on each player below once the cell finishes.

In [ ]:
import torch, torchaudio as ta
from chatterbox.tts import ChatterboxTTS
from IPython.display import Audio, display

print("Loading the voice model (first run downloads it)…")
cb = ChatterboxTTS.from_pretrained(device="cuda")

print("\nEnglish:")
en = cb.generate("This is ArcVox. Studio-grade voice, running on a GPU you control.")
ta.save("english.wav", en, cb.sr)
display(Audio("english.wav"))

print("\nHindi:")
hi = cb.generate("नमस्ते! यह आपकी अपनी मशीन पर चल रहा है। आपका डेटा कहीं नहीं जाता।")
ta.save("hindi.wav", hi, cb.sr)
display(Audio("hindi.wav"))

print("\nDone — press play above to listen.")

## ⭐ Now score it — this is the whole point

Listen to both clips above and rate the voice **1–10 against ElevenLabs**, as if you were a paying customer comparing two tabs. Be honest.

- **8–10** → the product thesis is **proven**. Stop adding features; go find a customer. (See `docs/DECISION_TREE.md` → 'scores 8–10' and `docs/GO_TO_MARKET.md`.)
- **5–7** → close but not sellable. Try `TTS_ENGINE` alternatives, then consider fine-tuning. (See `docs/DECISION_TREE.md` → 'scores 5–7' and `docs/FINETUNING.md`.)
- **1–4** → the base model is wrong, not your code. Swap engines and re-run. (See `docs/DECISION_TREE.md` → 'scores 1–4'.)

**Write the number down before moving on.** Your next move is pre-decided for each score.

In [ ]:
# free memory before the next step
import gc
del cb; gc.collect(); torch.cuda.empty_cache()
print("ok")

### Step 3 — transcribe what it just said (Whisper large-v3)
This is the engine that turns speech back into text — including Indian languages.

In [ ]:
!pip install -q faster-whisper

In [ ]:
from faster_whisper import WhisperModel
print("Loading the transcription model…")
asr = WhisperModel("large-v3", device="cuda", compute_type="float16")

for path in ["english.wav", "hindi.wav"]:
    segs, info = asr.transcribe(path, vad_filter=True)
    text = " ".join(s.text.strip() for s in segs)
    print(f"\n{path}  (detected language: {info.language})")
    print("  >>", text)

## What you just experienced

- The **English** and **Hindi** clips are real output from ArcVox's voice engine
  (Chatterbox, MIT-licensed) — **no model was trained**, these are pretrained
  weights running on the free GPU.
- The transcription read both clips back correctly — that's Whisper `large-v3`,
  ArcVox's transcription engine, which also handles Tamil, Telugu, Bengali, etc.

**If the voices sound good, the core product is proven.** Next steps when you're
ready (in `arcvox_gpu_verify.ipynb`): clone your own voice from a 30-second clip,
make a talking-head video from a photo, and hear all 21 Indian languages.

> Reminder: running here uses Google's servers — perfect for testing, but for
> real customers run these same engines on hardware you control.
